<a href="https://colab.research.google.com/github/ardizzone88/motor-inteligente-de-precios/blob/master/monitor_precios_competencia_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Monitor Inteligente de Precios y Competencia
### Dynamic Pricing Analytics para Sellers de E-commerce

**Autor:** David | Data Analyst Portfolio Project
**Stack:** Python · Pandas · Plotly · Scikit-Learn (K-Means, Regresión Lineal) · Best Buy Products API

---

## 📋 El problema de negocio

Todos los días, miles de vendedores de e-commerce fijan un precio y se olvidan de él. Mientras tanto, la competencia:

- 📉 Baja precios agresivamente para liquidar stock
- 🚫 Se queda sin inventario y deja la cancha libre (oportunidad perdida si no la detectás a tiempo)
- 🎯 Ajusta su oferta varias veces por día con herramientas que el vendedor chico no puede pagar

Este notebook construye, de punta a punta, el motor analítico de un **producto SaaS de Dynamic Pricing**: un sistema que monitorea a la competencia, detecta quiebres de stock y bajadas de precio, estima la elasticidad precio-demanda, segmenta a los competidores por comportamiento y calcula un **Precio Óptimo sugerido** — todo lo que hoy solo pueden pagar las grandes cadenas.

> ⚠️ **Nota sobre la fuente de datos:** la idea original de este proyecto era consumir la API pública de MercadoLibre. Sin embargo, **desde abril de 2025 MercadoLibre cerró el acceso público a su endpoint de búsqueda**: cualquier consulta sin un token OAuth de un usuario autorizado devuelve `403 forbidden`. Es un cambio de política real y documentado, no una limitación de este notebook.
>
> Para que el proyecto siga funcionando con **datos reales y verificables hoy**, se usa la **[Best Buy Products API](https://developer.bestbuy.com/)**: una API pública, gratuita, con key instantánea (sin OAuth ni aprobación de partner), que devuelve precio, disponibilidad y descuento *en tiempo real* sobre más de un millón de productos. El caso de estudio pasa de "vendedores compitiendo por el mismo ítem en un marketplace" a **"marcas competidoras dentro de una misma categoría de producto"** (ej. Sony vs. JBL vs. Bose en auriculares Bluetooth) — el mismo problema de negocio, la misma arquitectura analítica, con una fuente de datos que sí está disponible públicamente.
>
> 💡 **Nota de transparencia metodológica:** la Best Buy API entrega el snapshot de precio/stock *actual* de cada producto, no su historial. Para poder analizar tendencias (bajadas de precio, quiebres de stock en el tiempo) este notebook ancla el día de hoy a precios reales obtenidos vía API y simula 14 días de historial hacia atrás — el mismo enfoque que usaría el producto real corriendo un job programado que guarda cada consulta en una base de datos. Esto se indica explícitamente en cada sección.


## ⚙️ 0. Setup del entorno

In [1]:
# Instalación de librerías (en Colab, plotly y sklearn ya vienen preinstalados)
!pip install requests --quiet

import requests
import pandas as pd
import numpy as np
import json
import time
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression

np.random.seed(42)
pd.set_option('display.max_columns', None)

print("Entorno listo ✅")


Entorno listo ✅


## 1️⃣ Recolección de datos — Best Buy Products API (real)

La [Best Buy Developer API](https://developer.bestbuy.com/) es pública y gratuita: te registrás con tu email y te entrega una API key al instante (sin OAuth, sin aprobación de partner, sin esperar días). Devuelve precio regular, precio con descuento, si el producto está en oferta (`onSale`) y disponibilidad online/en tienda — todo en tiempo real sobre el catálogo vigente de Best Buy.

**Para correr esta celda con datos 100% reales:**
1. Entrá a https://developer.bestbuy.com/ → "Get API Key" (gratis, instantáneo).
2. Pegá tu key en la variable `BESTBUY_API_KEY` de la celda siguiente.

Si dejás la key en blanco (o la API no responde), el notebook cae automáticamente al generador sintético de la sección 2 — el resto del análisis funciona exactamente igual en ambos casos.

In [2]:
BESTBUY_API_KEY = ""  # 👈 Pegá aquí tu API key gratuita de https://developer.bestbuy.com/

CATEGORIA = "bluetooth headphones"  # Categoría/búsqueda real dentro del catálogo de Best Buy
LIMIT = 20

def consultar_bestbuy(query, api_key, limit=20):
    """Consulta la API real de Best Buy y devuelve un DataFrame con el snapshot actual de precio/stock."""
    if not api_key:
        print("ℹ️ No se configuró BESTBUY_API_KEY. Se usará un dataset sintético equivalente.")
        return None

    url = f"https://api.bestbuy.com/v1/products(search={query})"
    params = {
        "apiKey": api_key,
        "format": "json",
        "pageSize": limit,
        "show": "sku,name,manufacturer,regularPrice,salePrice,onSale,onlineAvailability,inStoreAvailability,customerReviewAverage,customerReviewCount",
    }

    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        registros = []
        for item in data.get("products", []):
            registros.append({
                "sku": item.get("sku"),
                "titulo": item.get("name"),
                "marca": item.get("manufacturer"),
                "precio": item.get("salePrice", item.get("regularPrice")),
                "precio_regular": item.get("regularPrice"),
                "en_oferta": item.get("onSale"),
                "disponible_online": item.get("onlineAvailability"),
                "disponible_tienda": item.get("inStoreAvailability"),
                "rating": item.get("customerReviewAverage"),
            })

        df = pd.DataFrame(registros)
        print(f"✅ {len(df)} productos obtenidos en tiempo real desde la API de Best Buy")
        return df

    except Exception as e:
        print(f"⚠️ No se pudo consultar la API real ({e}). Se usará un dataset sintético equivalente.")
        return None

df_api = consultar_bestbuy(CATEGORIA, BESTBUY_API_KEY, LIMIT)
if df_api is not None:
    display(df_api.head(10))


ℹ️ No se configuró BESTBUY_API_KEY. Se usará un dataset sintético equivalente.


## 2️⃣ Generador de datos sintéticos (fallback + simulación de historial)

Usamos esta función en dos casos:
1. Si no se configuró una API key real, o la API no respondió (rate limit, sin conexión, cambios de esquema).
2. Siempre, para **simular los 14 días de historial** de precio y stock que la API no entrega, anclando el día 14 (hoy) a los precios reales si están disponibles.

In [4]:
import os

COMPETIDORES_DEFAULT = ["Sony", "JBL", "Bose", "Apple", "Beats", "Skullcandy"]

def generar_dataset_sintetico(df_real=None, dias=14, n_competidores=6):
    """Genera un historial de 14 días de precio/stock por competidor, realista y reproducible.
    Si hay datos reales de la API de Best Buy, usa las marcas y precios reales como ancla."""

    if df_real is not None and df_real["marca"].nunique() >= n_competidores:
        precios_reales = df_real.groupby("marca")["precio"].mean().sort_values(ascending=False)
        competidores = precios_reales.index[:n_competidores].tolist()
        precios_base = precios_reales.values[:n_competidores]
        print(f"📡 Usando {n_competidores} marcas reales (anclaje de precio real, vía API): {', '.join(competidores)}")
    else:
        competidores = COMPETIDORES_DEFAULT[:n_competidores]
        precios_base = np.random.uniform(40, 350, n_competidores)  # rango realista de auriculares en USD

    registros = []
    hoy = datetime.now()

    for i, competidor in enumerate(competidores):
        precio_actual = precios_base[i]
        stock_actual = np.random.randint(8, 40)
        # Cada competidor tiene una "personalidad": agresivo, estable o premium
        personalidad = np.random.choice(["agresivo", "estable", "premium"], p=[0.35, 0.4, 0.25])
        # Día "evento" garantizado por competidor agresivo: una liquidación fuerte de precio
        dia_liquidacion = np.random.randint(3, dias - 1) if personalidad == "agresivo" else None

        for d in range(dias):
            fecha = hoy - timedelta(days=(dias - 1 - d))

            # Variación de precio según personalidad
            if personalidad == "agresivo":
                shock = np.random.normal(-0.015, 0.05)
                if d == dia_liquidacion:
                    shock = -np.random.uniform(0.10, 0.18)  # liquidación: -10% a -18% en un día
            elif personalidad == "premium":
                shock = np.random.normal(0.005, 0.01)
            else:
                shock = np.random.normal(0, 0.02)

            precio_actual = max(precio_actual * (1 + shock), precios_base[i] * 0.5)

            # Ventas del día (más ventas si el precio es competitivo, y un pico tras liquidar)
            precio_rel = precio_actual / np.mean(precios_base)
            lam_ventas = max(0.5, 7 - precio_rel * 4.5)
            ventas_dia = np.random.poisson(lam_ventas)
            stock_actual = max(0, stock_actual - ventas_dia)

            # Reposición ocasional (no garantizada: así puede llegar a quiebre real de stock)
            if stock_actual < 4 and np.random.rand() < 0.25:
                stock_actual += np.random.randint(15, 40)

            registros.append({
                "fecha": fecha.date(),
                "competidor": competidor,
                "personalidad": personalidad,
                "precio": round(precio_actual, 2),
                "stock_disponible": stock_actual,
                "vendidos_dia": ventas_dia,
            })

    return pd.DataFrame(registros)

df_historico = generar_dataset_sintetico(df_api, dias=14, n_competidores=6)
os.makedirs('data', exist_ok=True) # Create the directory if it doesn't exist
df_historico.to_csv("data/historico_precios_competencia.csv", index=False)
df_historico.tail(10)

,fecha,competidor,personalidad,precio,stock_disponible,vendidos_dia
74,2026-06-09,Skullcandy,premium,63.42,12,7
75,2026-06-10,Skullcandy,premium,63.14,10,2
76,2026-06-11,Skullcandy,premium,63.66,4,6
77,2026-06-12,Skullcandy,premium,63.18,0,5
78,2026-06-13,Skullcandy,premium,63.16,0,5
79,2026-06-14,Skullcandy,premium,63.72,37,6
80,2026-06-15,Skullcandy,premium,64.65,33,4
81,2026-06-16,Skullcandy,premium,65.32,30,3
82,2026-06-17,Skullcandy,premium,65.06,22,8
83,2026-06-18,Skullcandy,premium,64.79,19,3


## 3️⃣ Limpieza y preparación de variables

In [5]:
df = df_historico.copy()
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values(["competidor", "fecha"]).reset_index(drop=True)

# Variación diaria de precio por competidor (clave para detectar bajadas agresivas)
df["precio_dia_anterior"] = df.groupby("competidor")["precio"].shift(1)
df["variacion_pct"] = ((df["precio"] - df["precio_dia_anterior"]) / df["precio_dia_anterior"]) * 100

# Acumulado de ventas
df["vendidos_acum"] = df.groupby("competidor")["vendidos_dia"].cumsum()

# Flag de quiebre de stock
df["quiebre_stock"] = df["stock_disponible"] == 0

print(f"Dataset limpio: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Competidores monitoreados: {df['competidor'].nunique()} | Período: {df['fecha'].min().date()} → {df['fecha'].max().date()}")
df.head()


Dataset limpio: 84 filas x 10 columnas
Competidores monitoreados: 6 | Período: 2026-06-05 → 2026-06-18


,fecha,competidor,personalidad,precio,stock_disponible,vendidos_dia,precio_dia_anterior,variacion_pct,vendidos_acum,quiebre_stock
0,2026-06-05,Apple,estable,125.58,6,2,NaN,NaN,2,False
1,2026-06-06,Apple,estable,121.41,6,0,125.58,-3.320592,2,False
2,2026-06-07,Apple,estable,119.32,4,2,121.41,-1.721440,4,False
3,2026-06-08,Apple,estable,118.40,0,5,119.32,-0.771036,9,True
4,2026-06-09,Apple,estable,119.77,22,5,118.40,1.157095,14,False


## 4️⃣ Panorama competitivo — Evolución de precios

Visualizamos cómo se movió cada competidor durante las dos últimas semanas.

In [6]:
fig = px.line(df, x="fecha", y="precio", color="competidor",
              title="Evolución de precios por competidor (14 días)",
              labels={"precio": "Precio ($)", "fecha": "Fecha"},
              markers=True)
fig.update_layout(template="plotly_white", legend_title="Marca", height=500)
fig.show()


## 5️⃣ Detección de quiebres de stock ⚠️

Cuando un competidor se queda sin stock, abre una ventana de oportunidad: la demanda no desaparece, se redirige. Detectarlo rápido permite ajustar precio o stock propio para capturarla.

In [7]:
quiebres = df[df["quiebre_stock"]].groupby("competidor").size().reset_index(name="dias_sin_stock")
quiebres = quiebres.sort_values("dias_sin_stock", ascending=False)

print("📦 Resumen de quiebres de stock en los últimos 14 días:")
print(quiebres.to_string(index=False) if len(quiebres) > 0 else "Sin quiebres de stock detectados en el período.")

fig = px.bar(df.groupby(["fecha"])["quiebre_stock"].sum().reset_index(),
             x="fecha", y="quiebre_stock",
             title="Competidores sin stock por día",
             labels={"quiebre_stock": "Cantidad de competidores sin stock"})
fig.update_layout(template="plotly_white", height=400)
fig.show()


📦 Resumen de quiebres de stock en los últimos 14 días:
competidor  dias_sin_stock
     Beats               5
Skullcandy               2
      Sony               2
     Apple               1


## 6️⃣ Detección de bajadas agresivas de precio 📉

Definimos como "bajada agresiva" cualquier caída de precio mayor al **8% en un solo día** respecto al día anterior — un umbral típico que indica liquidación de stock o reacción defensiva, no ajuste fino.

In [8]:
UMBRAL_AGRESIVO = -8.0  # porcentaje

alertas_precio = df[df["variacion_pct"] <= UMBRAL_AGRESIVO][
    ["fecha", "competidor", "precio_dia_anterior", "precio", "variacion_pct"]
].copy()
alertas_precio["variacion_pct"] = alertas_precio["variacion_pct"].round(1)

print(f"🚨 Se detectaron {len(alertas_precio)} bajadas agresivas de precio en el período:")
alertas_precio.sort_values("fecha")


🚨 Se detectaron 1 bajadas agresivas de precio en el período:


,fecha,competidor,precio_dia_anterior,precio,variacion_pct
53,2026-06-16,JBL,143.35,119.07,-16.9


## 7️⃣ Elasticidad precio-demanda (proxy)

Estimamos qué tan sensible es la demanda al precio comparando, entre competidores, el precio promedio contra el total vendido. Usamos una regresión log-log: la pendiente es directamente el **coeficiente de elasticidad**.

In [9]:
resumen_competidor = df.groupby("competidor").agg(
    precio_promedio=("precio", "mean"),
    total_vendido=("vendidos_dia", "sum"),
    volatilidad_stock=("stock_disponible", "std"),
).reset_index()

resumen_competidor["total_vendido"] = resumen_competidor["total_vendido"].replace(0, 1)  # evitar log(0)

X = np.log(resumen_competidor[["precio_promedio"]])
y = np.log(resumen_competidor["total_vendido"])

modelo_elasticidad = LinearRegression().fit(X, y)
elasticidad = modelo_elasticidad.coef_[0]

print(f"📊 Coeficiente de elasticidad estimado: {elasticidad:.2f}")
if elasticidad < -1:
    interpretacion = "Demanda ELÁSTICA: los clientes de este producto son muy sensibles al precio. Compiten principalmente por precio."
elif elasticidad < 0:
    interpretacion = "Demanda INELÁSTICA: el precio influye, pero hay margen para diferenciarse por marca, envío o servicio."
else:
    interpretacion = "Relación atípica (posible efecto de marca/calidad dominando sobre precio) — revisar con más datos."

print(f"➡️ {interpretacion}")


📊 Coeficiente de elasticidad estimado: -1.40
➡️ Demanda ELÁSTICA: los clientes de este producto son muy sensibles al precio. Compiten principalmente por precio.


## 8️⃣ Segmentación de competidores — K-Means

No todos los competidores juegan el mismo juego. Agrupamos a los competidores por su comportamiento (precio promedio, volatilidad de precio, confiabilidad de stock y volumen de ventas) para identificar **arquetipos de competencia**.

In [10]:
features_cluster = df.groupby("competidor").agg(
    precio_promedio=("precio", "mean"),
    volatilidad_precio=("variacion_pct", "std"),
    dias_sin_stock=("quiebre_stock", "sum"),
    total_vendido=("vendidos_dia", "sum"),
).reset_index().fillna(0)

X_cluster = features_cluster[["precio_promedio", "volatilidad_precio", "dias_sin_stock", "total_vendido"]]
X_scaled = StandardScaler().fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
features_cluster["cluster"] = kmeans.fit_predict(X_scaled)

etiquetas = {
    features_cluster.groupby("cluster")["precio_promedio"].mean().idxmax(): "Premium / Diferenciado",
    features_cluster.groupby("cluster")["volatilidad_precio"].mean().idxmax(): "Guerra de Precios",
}
features_cluster["segmento"] = features_cluster["cluster"].map(
    lambda c: etiquetas.get(c, "Estable / Nicho")
)

fig = px.scatter(features_cluster, x="precio_promedio", y="volatilidad_precio",
                  color="segmento", size="total_vendido", text="competidor",
                  title="Segmentación de competidores por comportamiento",
                  labels={"precio_promedio": "Precio promedio ($)", "volatilidad_precio": "Volatilidad de precio (%)"})
fig.update_traces(textposition="top center")
fig.update_layout(template="plotly_white", height=500)
fig.show()

features_cluster[["competidor", "segmento", "precio_promedio", "dias_sin_stock", "total_vendido"]]


,competidor,segmento,precio_promedio,dias_sin_stock,total_vendido
0,Apple,Estable / Nicho,121.662857,1,45
1,Beats,Estable / Nicho,75.405000,5,57
2,Bose,Premium / Diferenciado,242.635714,0,8
3,JBL,Guerra de Precios,149.600000,0,26
4,Skullcandy,Estable / Nicho,63.667857,2,60
5,Sony,Estable / Nicho,134.567857,2,43


## 9️⃣ Motor de Precio Óptimo 🎯

El corazón del producto: dado el costo del vendedor (el usuario del SaaS), un margen mínimo aceptable y el panorama competitivo, ¿cuál es el precio que maximiza ganancia sin quedar fuera de mercado?

In [11]:
def calcular_precio_optimo(costo_propio, margen_minimo, df_competencia, elasticidad_estimada):
    """
    Sugiere un precio óptimo considerando:
    - el piso de margen del vendedor (no vender a pérdida)
    - el precio mínimo de la competencia activa (con stock)
    - si la demanda es elástica o no, para decidir cuánto acercarse al mínimo competidor
    """
    ultimo_dia = df_competencia["fecha"].max()
    snapshot = df_competencia[df_competencia["fecha"] == ultimo_dia]
    competencia_activa = snapshot[snapshot["stock_disponible"] > 0]

    # margen_minimo se interpreta como % de margen sobre el PRECIO DE VENTA (no markup sobre costo)
    precio_piso = costo_propio / (1 - margen_minimo)

    if competencia_activa.empty:
        return {"precio_sugerido": round(precio_piso, 2), "motivo": "Sin competencia con stock: se sugiere precio de margen objetivo."}

    precio_min_competencia = competencia_activa["precio"].min()

    if elasticidad_estimada < -1:
        # Demanda elástica → hay que ser agresivo en precio
        precio_sugerido = max(precio_piso, precio_min_competencia * 0.97)
        motivo = "Demanda elástica: se prioriza estar por debajo del mínimo competidor para ganar volumen."
    else:
        # Demanda inelástica → se puede cobrar cerca del promedio de mercado
        precio_promedio_mercado = competencia_activa["precio"].mean()
        precio_sugerido = max(precio_piso, min(precio_promedio_mercado, precio_min_competencia * 1.05))
        motivo = "Demanda inelástica: se sugiere un precio cercano al promedio de mercado, priorizando margen sobre volumen."

    return {
        "precio_sugerido": round(precio_sugerido, 2),
        "precio_min_competencia": round(precio_min_competencia, 2),
        "margen_resultante_pct": round((precio_sugerido - costo_propio) / precio_sugerido * 100, 1),
        "motivo": motivo,
    }

# Ejemplo: el vendedor (usuario del SaaS) tiene un costo de USD 80 y quiere al menos 20% de margen
resultado = calcular_precio_optimo(
    costo_propio=80,
    margen_minimo=0.20,
    df_competencia=df,
    elasticidad_estimada=elasticidad,
)
resultado


{'precio_sugerido': 100.0,
 'precio_min_competencia': 64.79,
 'margen_resultante_pct': 20.0,
 'motivo': 'Demanda elástica: se prioriza estar por debajo del mínimo competidor para ganar volumen.'}

## 🔔 10. Sistema de alertas (simulación de webhook a Telegram/Discord)

En producción, este paso se ejecuta cada vez que el job programado detecta una novedad y dispara una notificación al vendedor. Acá lo simulamos: si se provee un `bot_token` real, el mensaje se envía de verdad; si no, se imprime en consola.

In [12]:
def enviar_alerta_telegram(mensaje, bot_token=None, chat_id=None):
    if bot_token and chat_id:
        url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
        try:
            requests.post(url, data={"chat_id": chat_id, "text": mensaje}, timeout=10)
            print("✅ Alerta enviada por Telegram.")
        except Exception as e:
            print(f"⚠️ No se pudo enviar la alerta real ({e}). Mensaje simulado:")
            print(mensaje)
    else:
        print("📨 [ALERTA SIMULADA — configurar bot_token/chat_id para envío real]")
        print(mensaje)
    print("-" * 60)

# Generamos las alertas del día a partir de lo detectado arriba
for _, fila in alertas_precio.iterrows():
    msg = f"🚨 {fila['competidor']} bajó su precio un {abs(fila['variacion_pct'])}% el {fila['fecha'].date()}. Evaluar respuesta."
    enviar_alerta_telegram(msg)

for _, fila in quiebres.iterrows():
    msg = f"📦 {fila['competidor']} estuvo sin stock {fila['dias_sin_stock']} día(s) en las últimas 2 semanas. Oportunidad de capturar demanda."
    enviar_alerta_telegram(msg)

enviar_alerta_telegram(
    f"🎯 Precio óptimo sugerido para tu producto: ${resultado['precio_sugerido']} "
    f"(margen estimado: {resultado['margen_resultante_pct']}%). Motivo: {resultado['motivo']}"
)


📨 [ALERTA SIMULADA — configurar bot_token/chat_id para envío real]
🚨 JBL bajó su precio un 16.9% el 2026-06-16. Evaluar respuesta.
------------------------------------------------------------
📨 [ALERTA SIMULADA — configurar bot_token/chat_id para envío real]
📦 Beats estuvo sin stock 5 día(s) en las últimas 2 semanas. Oportunidad de capturar demanda.
------------------------------------------------------------
📨 [ALERTA SIMULADA — configurar bot_token/chat_id para envío real]
📦 Skullcandy estuvo sin stock 2 día(s) en las últimas 2 semanas. Oportunidad de capturar demanda.
------------------------------------------------------------
📨 [ALERTA SIMULADA — configurar bot_token/chat_id para envío real]
📦 Sony estuvo sin stock 2 día(s) en las últimas 2 semanas. Oportunidad de capturar demanda.
------------------------------------------------------------
📨 [ALERTA SIMULADA — configurar bot_token/chat_id para envío real]
📦 Apple estuvo sin stock 1 día(s) en las últimas 2 semanas. Oportunidad de

## 📊 11. Dashboard final — Panel de control del vendedor

Consolidamos todo en un panel único, el tipo de vista que vería un cliente del SaaS al loguearse.

In [13]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Evolución de precios (14 días)", "Quiebres de stock por día",
                     "Segmentación de competidores", "Ventas acumuladas por competidor"),
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "xy"}, {"type": "xy"}]]
)

for competidor in df["competidor"].unique():
    d = df[df["competidor"] == competidor]
    fig.add_trace(go.Scatter(x=d["fecha"], y=d["precio"], name=competidor, mode="lines"), row=1, col=1)

quiebres_dia = df.groupby("fecha")["quiebre_stock"].sum().reset_index()
fig.add_trace(go.Bar(x=quiebres_dia["fecha"], y=quiebres_dia["quiebre_stock"],
                      name="Sin stock", showlegend=False), row=1, col=2)

for seg in features_cluster["segmento"].unique():
    d = features_cluster[features_cluster["segmento"] == seg]
    fig.add_trace(go.Scatter(x=d["precio_promedio"], y=d["volatilidad_precio"],
                              mode="markers+text", text=d["competidor"], name=seg), row=2, col=1)

ventas_acum = df.groupby("competidor")["vendidos_dia"].sum().sort_values(ascending=False).reset_index()
fig.add_trace(go.Bar(x=ventas_acum["competidor"], y=ventas_acum["vendidos_dia"],
                      name="Vendidos", showlegend=False), row=2, col=2)

fig.update_layout(height=800, template="plotly_white",
                   title_text="📊 Dashboard — Monitor Inteligente de Precios y Competencia")
fig.show()


## ✅ 12. Conclusiones de negocio

- El mercado monitoreado muestra una elasticidad que indica si conviene competir por **precio** o por **diferenciación** (envío, marca, atención).
- Se identificaron ventanas de oportunidad claras por quiebres de stock de la competencia.
- El motor de Precio Óptimo traduce todo el análisis en **una sola recomendación accionable**, que es exactamente lo que un vendedor mediano necesita y no puede construir solo.
- Este notebook es el corazón analítico de un producto SaaS: lo que falta para llevarlo a producción es (a) un scheduler que lo corra cada N minutos, (b) persistencia en una base de datos (no CSV), y (c) un frontend simple o bot de Telegram como interfaz para el cliente.

➡️ Ver el **README** del repositorio para el modelo de negocio completo y las recomendaciones estratégicas.
